# 06 - Ask a small vision-language model "does a collision happen?" (Kaggle GPU)

**Idea:** Qwen2-VL-2B can look at several frames in order and answer a question. We do not read its text answer.
We look at the **probability it gives to "Yes" versus "No"** as the next word: `score = logit(Yes) - logit(No)`. High = it believes a collision happens.
It never saw our labels, so it cannot learn the train-set quirks that hurt our fitted models.

**Per clip:** 3 windows of 8 frames (whole clip, first half, second half), one forward pass each, no text generation.
**Output:** `qwen_scores.jsonl` (one line per clip: `split, clip_id, [score_all, score_first, score_second]`).
**Clips scored:** 200 train clips (100 whole pairs, only to check that the score ranks crash above its twin) + all 1750 test clips.

**Settings on Kaggle:** GPU T4 x2, Internet On (model download), competition attached. Weights are loaded in float32 because float16 on a T4 can produce NaNs with this model.
It starts one worker process per GPU. Local smoke test: `SMOKE=2`, `DEVICE=cpu`.

In [ ]:
import glob, os, sys, json, subprocess, time
import numpy as np
import pandas as pd
import torch

found = glob.glob("/kaggle/input/**/train_labels.csv", recursive=True) + glob.glob("../../data/train_labels.csv")
assert found, "train_labels.csv not found - attach the competition / check the path"
ROOT = os.path.dirname(found[0])
SMOKE = int(os.environ.get("SMOKE", 0)) or None
DEVICE = os.environ.get("DEVICE", "cuda")
on_kaggle = os.path.exists("/kaggle/working")
OUT = "/kaggle/working/features" if on_kaggle else ("../../artifacts/qwen_smoke" if SMOKE else "../../artifacts/qwen")
os.makedirs(OUT, exist_ok=True)

lab = pd.read_csv(f"{ROOT}/train_labels.csv")
test_ids = pd.read_csv(f"{ROOT}/sample_submission.csv")["clip_id"].tolist()
# 100 whole pairs from train (so we can check 'crash clip beats its twin'), then the whole test set
eval_groups = lab["group_id"].drop_duplicates().tolist()[:100]
train_items = [("train", c) for c in lab[lab.group_id.isin(eval_groups)]["clip_id"]]
items = train_items + [("test", c) for c in test_ids]
if SMOKE:
    items = train_items[:SMOKE] + [("test", c) for c in test_ids[:SMOKE]]
n_gpu = 1 if DEVICE == "cpu" else max(1, torch.cuda.device_count())
print(ROOT, "| clips to score:", len(items), "| workers:", n_gpu)

## The worker script
One copy runs per GPU. For each clip it builds 3 windows of 8 frames, asks the question, and reads the Yes/No logits of the last position.
Results are appended line by line, so nothing is lost if the session stops.

In [ ]:
WORKER = r"""
import sys, os, json
import numpy as np, torch
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

root, items_file, out_file, device = sys.argv[1:5]
M = "Qwen/Qwen2-VL-2B-Instruct"
model = Qwen2VLForConditionalGeneration.from_pretrained(M, torch_dtype=torch.float32, attn_implementation="eager").to(device).eval()
proc = AutoProcessor.from_pretrained(M, min_pixels=224 * 392, max_pixels=224 * 392)   # 384x224 frames -> 392x224 -> 112 visual tokens each
tok = proc.tokenizer
yes_ids = sorted({tok.encode(w, add_special_tokens=False)[0] for w in ["Yes", "yes", " Yes"]})
no_ids = sorted({tok.encode(w, add_special_tokens=False)[0] for w in ["No", "no", " No"]})

QUESTION = ("These are consecutive frames from a car dashcam, in time order. Does a traffic collision happen in these frames - "
            "the camera car or another vehicle hitting a vehicle, person or object? Answer Yes or No.")
WINDOWS = [np.linspace(0, 29, 8), np.linspace(0, 15, 8), np.linspace(14, 29, 8)]   # whole clip, first half, second half

def lse(x):
    return torch.logsumexp(x, dim=0)

done = set()
if os.path.exists(out_file):
    for line in open(out_file):
        r = json.loads(line); done.add((r["split"], r["clip_id"]))
items = [tuple(x) for x in json.load(open(items_file))]
with open(out_file, "a") as f:
    for k, (split, cid) in enumerate(items):
        if (split, cid) in done:
            continue
        frames = [Image.open(f"{root}/{split}/{cid}/frame_{j:03d}.jpg").convert("RGB") for j in range(30)]
        scores = []
        for w in WINDOWS:
            imgs = [frames[int(round(i))] for i in w]
            msg = [{"role": "user", "content": [{"type": "image"} for _ in imgs] + [{"type": "text", "text": QUESTION}]}]
            text = proc.apply_chat_template(msg, add_generation_prompt=True)
            inputs = proc(text=[text], images=imgs, return_tensors="pt").to(device)
            with torch.no_grad():
                logits = model(**inputs).logits[0, -1].float()
            scores.append(float(lse(logits[yes_ids]) - lse(logits[no_ids])))
        f.write(json.dumps({"split": split, "clip_id": cid, "scores": scores}) + "\n"); f.flush()
        if k % 50 == 0:
            print(k, len(items), scores, flush=True)
"""
open("worker.py", "w").write(WORKER)

## Split the clips over the GPUs and run the workers
Each worker gets every n-th clip; each runs on its own GPU through `CUDA_VISIBLE_DEVICES`. We wait for both to finish.

In [ ]:
procs, out_files = [], []
for g in range(n_gpu):
    shard = items[g::n_gpu]
    json.dump(shard, open(f"items_{g}.json", "w"))
    out_file = f"{OUT}/qwen_scores_{g}.jsonl"
    out_files.append(out_file)
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g)) if DEVICE != "cpu" else dict(os.environ)
    dev = "cpu" if DEVICE == "cpu" else "cuda"
    log = open(f"worker_{g}.log", "w")
    procs.append(subprocess.Popen([sys.executable, "worker.py", ROOT, f"items_{g}.json", out_file, dev], env=env, stdout=log, stderr=subprocess.STDOUT))
t0 = time.time()
for p in procs:
    p.wait()
print("workers finished in", f"{time.time() - t0:.0f}s", "| return codes:", [p.returncode for p in procs])
for g in range(n_gpu):
    print(f"--- worker {g} log tail ---"); print("".join(open(f"worker_{g}.log").readlines()[-5:]))

## Merge the shards
One file with all clips and one row per clip. Check for missing clips and non-finite scores.

In [ ]:
rows = []
for fpath in out_files:
    if os.path.exists(fpath):
        rows += [json.loads(l) for l in open(fpath)]
df = pd.DataFrame([{"split": r["split"], "clip_id": r["clip_id"], "all": r["scores"][0], "first": r["scores"][1], "second": r["scores"][2]} for r in rows])
df.to_csv(f"{OUT}/qwen_scores.csv", index=False)
print(df.shape, "| expected:", len(items), "| non-finite:", int((~np.isfinite(df[["all", "first", "second"]].values)).sum()))
print(df.groupby("split")[["all", "first", "second"]].mean().round(2))